# 00 – Data Profiling and Load

**Purpose:** Profile the raw Favorita files and load them into DuckDB with correct types.

**Inputs:** `data/raw/*.csv` (Kaggle Favorita) + `data/raw/weather_daily.csv` (Open-Meteo)
**Outputs:** tables `sales` (125.5M rows), `items`, `stores`, `oil`, `holidays_events`, `transactions`, `weather`

**Key decisions:**
- Grain and keys proven with counts, not assumed from samples
- `id` dropped; business key = date + store + item
- NULL promo flags kept as NULL (unknown ≠ false)
- Naming: lowercase, units in names, `is_` prefix for flags

In [40]:
import duckdb
con = duckdb.connect('../data/favorita.duckdb')   # creates the file on first run
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

In [2]:
con.sql("""
DESCRIBE SELECT * FROM read_csv('../data/raw/stores.csv')
""").df()

,column_name,column_type,null,key,default,extra
0,store_nbr,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,state,VARCHAR,YES,None,None,None
3,type,VARCHAR,YES,None,None,None
4,cluster,BIGINT,YES,None,None,None


In [3]:
con.sql("""
SELECT * FROM read_csv('../data/raw/stores.csv') LIMIT 5
""").df()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [5]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/items.csv')""").df()

,column_name,column_type,null,key,default,extra
0,item_nbr,BIGINT,YES,None,None,None
1,family,VARCHAR,YES,None,None,None
2,class,BIGINT,YES,None,None,None
3,perishable,BIGINT,YES,None,None,None


In [6]:
con.sql("""SELECT * FROM read_csv('../data/raw/items.csv')""").df()

,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1
...,...,...,...,...
4095,2132318,GROCERY I,1002,0
4096,2132945,GROCERY I,1026,0
4097,2132957,GROCERY I,1068,0
4098,2134058,BEVERAGES,1124,0


In [7]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/oil.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,dcoilwtico,DOUBLE,YES,None,None,None


In [8]:
con.sql("""SELECT * FROM read_csv('../data/raw/oil.csv')""").df()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
...,...,...
1213,2017-08-25,47.65
1214,2017-08-28,46.40
1215,2017-08-29,46.46
1216,2017-08-30,45.96


In [10]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/holidays_events.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,type,VARCHAR,YES,None,None,None
2,locale,VARCHAR,YES,None,None,None
3,locale_name,VARCHAR,YES,None,None,None
4,description,VARCHAR,YES,None,None,None
5,transferred,BOOLEAN,YES,None,None,None


In [11]:
con.sql("""
SELECT * FROM read_csv('../data/raw/holidays_events.csv') LIMIT 5
""").df()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [12]:
con.sql("""
SUMMARIZE SELECT * FROM read_csv('../data/raw/holidays_events.csv')
""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,date,DATE,2012-03-02,2017-12-26,315,2015-04-24 00:45:15.428571,<NA>,2013-12-22,2015-06-05,2016-07-09,350,0.0
1,type,VARCHAR,Additional,Work Day,6,NaN,<NA>,NaN,NaN,NaN,350,0.0
2,locale,VARCHAR,Local,Regional,3,NaN,<NA>,NaN,NaN,NaN,350,0.0
3,locale_name,VARCHAR,Ambato,Santo Domingo de los Tsachilas,22,NaN,<NA>,NaN,NaN,NaN,350,0.0
4,description,VARCHAR,Batalla de Pichincha,Viernes Santo,110,NaN,<NA>,NaN,NaN,NaN,350,0.0
5,transferred,BOOLEAN,false,true,2,NaN,<NA>,NaN,NaN,NaN,350,0.0


In [13]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/transactions.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,BIGINT,YES,None,None,None
2,transactions,BIGINT,YES,None,None,None


In [14]:
con.sql("""
    SELECT * FROM read_csv('../data/raw/transactions.csv') LIMIT 5""").df()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [15]:
con.sql("""
SUMMARIZE SELECT * FROM read_csv('../data/raw/transactions.csv')
""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,date,DATE,2013-01-01,2017-08-15,1727,2015-05-20 16:07:40.866232,NaN,2014-03-27,2015-06-08,2016-07-15,83488,0.0
1,store_nbr,BIGINT,1,54,49,26.939236776542735,15.608203645366135,13,27,40,83488,0.0
2,transactions,BIGINT,5,8359,4976,1694.6021583940208,963.2866435167193,1046,1393,2082,83488,0.0


In [16]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/train.csv')""").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,date,DATE,YES,None,None,None
2,store_nbr,BIGINT,YES,None,None,None
3,item_nbr,BIGINT,YES,None,None,None
4,unit_sales,DOUBLE,YES,None,None,None
5,onpromotion,VARCHAR,YES,None,None,None


In [18]:
con.sql("""
    SELECT * FROM read_csv('../data/raw/train.csv') WHERE onpromotion IS NOT NULL LIMIT 5""").df()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,21779262,2014-04-03,18,1239795,1.0,False
1,21779263,2014-04-03,18,1239796,6.0,False
2,21779264,2014-04-03,18,1239800,3.0,False
3,21779265,2014-04-03,18,1239806,2.0,False
4,21779266,2014-04-03,18,1239807,1.0,False


In [19]:
con.sql("""
SUMMARIZE SELECT * FROM read_csv('../data/raw/train.csv')
""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,id,BIGINT,0,125497039,131825810,62748519.5,36227875.05758848,31458204,62755454,94140410,125497040,0.00
1,date,DATE,2013-01-01,2017-08-15,1727,2015-09-15 16:43:18.78925,NaN,2014-09-21,2015-11-20,2016-10-17,125497040,0.00
2,store_nbr,BIGINT,1,54,49,27.464578025107205,16.330510302719947,12,28,43,125497040,0.00
3,item_nbr,BIGINT,96995,2127114,4092,972769.152722598,520533.59861073724,531133,965357,1354469,125497040,0.00
4,unit_sales,DOUBLE,-15372.0,89440.0,301723,8.554865268437917,23.605151809802596,2.0,4.000016027905985,8.96112516375932,125497040,0.00
5,onpromotion,VARCHAR,False,True,2,NaN,NaN,NaN,NaN,NaN,125497040,17.26


# Loading data in tables with right data type

** Create table items

In [20]:
con.sql("""
CREATE OR REPLACE TABLE items AS
SELECT
    item_nbr::INTEGER      AS item_nbr,
    family,
    class::INTEGER         AS class,
    perishable::BOOLEAN    AS is_perishable
FROM read_csv('../data/raw/items.csv')
""")

In [21]:
con.sql("DESCRIBE items").df()

,column_name,column_type,null,key,default,extra
0,item_nbr,INTEGER,YES,None,None,None
1,family,VARCHAR,YES,None,None,None
2,class,INTEGER,YES,None,None,None
3,is_perishable,BOOLEAN,YES,None,None,None


** Create Table stores

In [22]:
con.sql("""
DESCRIBE SELECT * FROM read_csv('../data/raw/stores.csv')
""").df()

,column_name,column_type,null,key,default,extra
0,store_nbr,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,state,VARCHAR,YES,None,None,None
3,type,VARCHAR,YES,None,None,None
4,cluster,BIGINT,YES,None,None,None


In [30]:
con.sql("""CREATE OR REPLACE TABLE stores AS
            SELECT 
            store_nbr::INTEGER AS store_nbr,
            city,
            state,
            type AS store_type,
            cluster::INTEGER AS cluster
            FROM read_csv('../data/raw/stores.csv')""")

In [31]:
con.sql("""DESCRIBE SELECT * FROM stores""").df()

,column_name,column_type,null,key,default,extra
0,store_nbr,INTEGER,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,state,VARCHAR,YES,None,None,None
3,store_type,VARCHAR,YES,None,None,None
4,cluster,INTEGER,YES,None,None,None


In [32]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/oil.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,dcoilwtico,DOUBLE,YES,None,None,None


In [34]:
con.sql(""" CREATE OR REPLACE TABLE oil AS
            SELECT 
            date,
            dcoilwtico AS oil_price_wti
            FROM read_csv('../data/raw/oil.csv')""")

In [35]:
con.sql("""DESCRIBE SELECT * FROM oil""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,oil_price_wti,DOUBLE,YES,None,None,None


In [36]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/holidays_events.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,type,VARCHAR,YES,None,None,None
2,locale,VARCHAR,YES,None,None,None
3,locale_name,VARCHAR,YES,None,None,None
4,description,VARCHAR,YES,None,None,None
5,transferred,BOOLEAN,YES,None,None,None


In [6]:
con.sql("""CREATE OR REPLACE TABLE holidays_events AS 
            SELECT 
            date::DATE AS date,
            type AS holiday_type,
            locale,
            locale_name,
            description,
            transferred::BOOLEAN AS is_transferred
            FROM read_csv('../data/raw/holidays_events.csv')
            """)

In [5]:
con.sql("""SELECT * FROM holidays_events LIMIT 5""").df()

,date,holiday_type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [7]:
con.sql("DESCRIBE holidays_events").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,holiday_type,VARCHAR,YES,None,None,None
2,locale,VARCHAR,YES,None,None,None
3,locale_name,VARCHAR,YES,None,None,None
4,description,VARCHAR,YES,None,None,None
5,is_transferred,BOOLEAN,YES,None,None,None


In [8]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/transactions.csv')""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,BIGINT,YES,None,None,None
2,transactions,BIGINT,YES,None,None,None


In [15]:
con.sql("""CREATE OR REPLACE TABLE transactions AS
            SELECT
            date::DATE AS date,
            store_nbr::INTEGER AS store_nbr,
            transactions::INTEGER AS no_of_transactions
            FROM read_csv('../data/raw/transactions.csv')
            """)


In [16]:
con.sql("SHOW TABLES").df()

,name
0,holidays_events
1,items
2,oil
3,stores
4,transactions


In [17]:
con.sql("""
    DESCRIBE SELECT * FROM read_csv('../data/raw/train.csv')""").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,date,DATE,YES,None,None,None
2,store_nbr,BIGINT,YES,None,None,None
3,item_nbr,BIGINT,YES,None,None,None
4,unit_sales,DOUBLE,YES,None,None,None
5,onpromotion,VARCHAR,YES,None,None,None


In [19]:
con.sql("""CREATE OR REPLACE TABLE sales AS 
            SELECT 
            date::DATE AS date,
            store_nbr::INTEGER AS store_nbr,
            item_nbr::INTEGER AS item_nbr,
            unit_sales::DOUBLE AS unit_sales,
            onpromotion::BOOLEAN AS is_on_promotion
            FROM read_csv('../data/raw/train.csv')""")

In [20]:
con.sql("""DESCRIBE SELECT * FROM sales""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,INTEGER,YES,None,None,None
2,item_nbr,INTEGER,YES,None,None,None
3,unit_sales,DOUBLE,YES,None,None,None
4,is_on_promotion,BOOLEAN,YES,None,None,None


In [21]:
con.sql("""SELECT COUNT(*) FROM sales""").df()

,count_star()
0,125497040


** All tables are loaded in database, next step is to check the data quality 

In [22]:
con.close()

## Load weather – daily weather by store city

** Open-Meteo file as a table called weather in the database, with types that match each column’s meaning and names a planner would understand, including the unit of each measure. Also make sure it will join cleanly to sales (date) and stores (city) - Reference from 03_weather.ipynb

In [2]:
con.sql("""DESCRIBE SELECT * FROM read_csv('../data/raw/weather_daily.csv')""").df()

,column_name,column_type,null,key,default,extra
0,time,DATE,YES,None,None,None
1,temperature_2m_max,DOUBLE,YES,None,None,None
2,temperature_2m_min,DOUBLE,YES,None,None,None
3,temperature_2m_mean,DOUBLE,YES,None,None,None
4,precipitation_sum,DOUBLE,YES,None,None,None
5,city,VARCHAR,YES,None,None,None


In [16]:
con.sql("""CREATE OR REPLACE TABLE weather AS 
            SELECT
            city::VARCHAR AS city,
            time::DATE AS date,
            temperature_2m_max::DOUBLE AS temp_max_deg_c,
            temperature_2m_min::DOUBLE AS temp_min_deg_c,
            temperature_2m_mean::DOUBLE AS temp_mean_deg_c,
            precipitation_sum::DOUBLE AS precipitation_mm
            FROM read_csv('../data/raw/weather_daily.csv')""")


In [17]:
con.sql("""DESCRIBE SELECT * FROM weather""").df()

,column_name,column_type,null,key,default,extra
0,city,VARCHAR,YES,None,None,None
1,date,DATE,YES,None,None,None
2,temp_max_deg_c,DOUBLE,YES,None,None,None
3,temp_min_deg_c,DOUBLE,YES,None,None,None
4,temp_mean_deg_c,DOUBLE,YES,None,None,None
5,precipitation_mm,DOUBLE,YES,None,None,None


In [19]:
con.sql("SELECT COUNT(*) AS tot_no_of_rows FROM weather").df()

,tot_no_of_rows
0,11097


In [22]:
con.sql("""SELECT 
            city,
            date,
            COUNT(*) AS no_of_unique_comb
            FROM weather 
            Group By city, date 
            HAVING (no_of_unique_comb)>1""").df()

,city,date,no_of_unique_comb


** Weather Check – Do weather cities match store cities

In [38]:
con.sql("""
SELECT
    s.store_nbr,
    s.city
FROM stores s
LEFT JOIN weather w
    ON s.city = w.city
WHERE s.store_nbr IN (44, 3, 17, 39, 50, 38, 11, 51, 34, 28, 40, 43, 27)
  AND w.city IS NULL
""").df()

,store_nbr,city


In [41]:
con.close()